# Early diabetes readmission: Spark and AutoML
UCI Diabetes 130-US hospitals (1999–2008), variant 14. The original 101,766 encounters are allowed by the assignment; no synthetic expansion is used. The binary outcome is readmission within 30 days (`<30`), versus later/no readmission.
The earliest recorded encounter per patient is retained before splitting, so patients do not leak across training, CV or holdout. This changes the population to first observed encounters. Predictors are available at discharge; this is an educational benchmark, not a validated clinical tool.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.readmission import prepare, train, reload_check
data, counts = prepare()

root
 |-- encounter_id: integer (nullable = true)
 |-- patient_nbr: integer (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: string (nullable = true)
 |-- weight: string (nullable = true)
 |-- admission_type_id: integer (nullable = true)
 |-- discharge_disposition_id: integer (nullable = true)
 |-- admission_source_id: integer (nullable = true)
 |-- time_in_hospital: double (nullable = true)
 |-- payer_code: string (nullable = true)
 |-- medical_specialty: string (nullable = true)
 |-- num_lab_procedures: double (nullable = true)
 |-- num_procedures: double (nullable = true)
 |-- num_medications: double (nullable = true)
 |-- number_outpatient: double (nullable = true)
 |-- number_emergency: double (nullable = true)
 |-- number_inpatient: double (nullable = true)
 |-- diag_1: string (nullable = true)
 |-- diag_2: string (nullable = true)
 |-- diag_3: string (nullable = true)
 |-- number_diagnoses: double (nullable = true)
 |-- max_gl

+------------+-----------+---------+------+-------+---------+-----------------+------------------------+-------------------+----------------+----------+----------------------+------------------+--------------+---------------+-----------------+----------------+----------------+------+------+------+----------------+-------------+---------+---------+-----------+-----------+--------------+-----------+-------------+---------+---------+-----------+------------+-------------+--------+--------+------------+----------+-------+-----------+-------+-------------------+-------------------+------------------------+-----------------------+----------------------+------+-----------+----------+------------------+-------------------+-----------------+-----------------+
|encounter_id|patient_nbr|race     |gender|age    |weight   |admission_type_id|discharge_disposition_id|admission_source_id|time_in_hospital|payer_code|medical_specialty     |num_lab_procedures|num_procedures|num_medications|number_outpati

{'raw_rows': 101766, 'deduplicated_encounters': 101766, 'unique_patients': 71518}


## Training-only model selection
The external stratified 20% holdout is isolated before sampling 18,000 training patients. PyCaret fits imputation, categorical encoding and normalization within three-fold stratified CV. Two CPU searches each have a two-minute budget and three candidate models. Logistic regression is the simple baseline in the first search. The final choice uses CV AUC; the holdout is only for reporting.
`total_prior_visits`, `medications_per_day`, and `labs_per_day` are calculated in Spark. Encounter/patient IDs and original outcome are excluded. PyCaret finalization uses only the supplied training sample.

In [2]:
result = train(data, counts)
import pandas as pd
display(pd.DataFrame(result["runs"]))

Cross-validation leaderboard: linear-trees
                       Model  Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC  TT (Sec)
lr       Logistic Regression    0.9124  0.6531  0.0144  0.5871  0.0282  0.0241  0.0818    0.9667
rf  Random Forest Classifier    0.9121  0.6106  0.0063  0.5278  0.0125  0.0105  0.0501    0.4733
dt  Decision Tree Classifier    0.8285  0.5255  0.1578  0.1246  0.1392  0.0454  0.0458    0.2300


2026/10/23 01:37:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Cross-validation leaderboard: boosting-discriminant
                            Model  Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC  TT (Sec)
lda  Linear Discriminant Analysis    0.9054  0.6557  0.0640  0.3204  0.1055  0.0786  0.1084    0.2300
gbc  Gradient Boosting Classifier    0.9119  0.6546  0.0234  0.4843  0.0447  0.0370  0.0916    0.9033
ada          Ada Boost Classifier    0.9112  0.6470  0.0180  0.4204  0.0345  0.0271  0.0710    0.4833


2026/10/23 01:37:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{
  "counts": {
    "raw_rows": 101766,
    "deduplicated_encounters": 101766,
    "unique_patients": 71518,
    "training_pool": 57214,
    "automl_rows": 18000,
    "holdout_rows": 14304,
    "training_positives": 1584,
    "holdout_positives": 1259,
    "patient_overlap": 0
  },
  "versions": {
    "pycaret": "3.3.2",
    "pyspark": "3.5.9",
    "scikit-learn": "1.4.2",
    "pandas": "2.1.4",
    "mlflow": "2.22.5"
  },
  "runs": [
    {
      "run": "linear-trees",
      "run_id": "ea3597f96fbd4648badbddbe13a4af3d",
      "selected": "LogisticRegression",
      "cv_auc": 0.6531,
      "roc_auc": 0.6427090072215959,
      "accuracy": 0.9117030201342282,
      "f1": 0.01096319498825372,
      "pr_auc": 0.16753721124551718
    },
    {
      "run": "boosting-discriminant",
      "run_id": "6a41a0038d7f4bd9be6d7df29ba01285",
      "selected": "LinearDiscriminantAnalysis",
      "cv_auc": 0.6557,
      "roc_auc": 0.6455538672725407,
      "accuracy": 0.908417225950783,
      "f1": 0.085

,run,run_id,selected,cv_auc,roc_auc,accuracy,f1,pr_auc
0,linear-trees,ea3597f96fbd4648badbddbe13a4af3d,LogisticRegression,0.6531,0.642709,0.911703,0.010963,0.167537
1,boosting-discriminant,6a41a0038d7f4bd9be6d7df29ba01285,LinearDiscriminantAnalysis,0.6557,0.645554,0.908417,0.085196,0.172922


## Serialization and reproducibility
Each saved model includes preprocessing. Both processed data and model artifacts are tracked through DVC; MLflow stores parameters, versions, counts, leaderboards, metrics and model artifacts locally. ROC-AUC and average precision evaluate ranking; F1 and accuracy use the default 0.5 decision threshold, which was not adjusted on holdout.

In [3]:
probabilities = reload_check(result)
display(probabilities)

Reloaded full preprocessing pipeline: (10, 2)


array([[0.95847118, 0.04152882],
       [0.9671373 , 0.0328627 ],
       [0.87538357, 0.12461643],
       [0.95565477, 0.04434523],
       [0.9680699 , 0.0319301 ],
       [0.9394494 , 0.0605506 ],
       [0.93475577, 0.06524423],
       [0.94782258, 0.05217742],
       [0.78547396, 0.21452604],
       [0.87441516, 0.12558484]])